# Gemma 4 Developer Agent — Phase 0 checks, EXP-000 gold control, baseline eval (4x L4)

One notebook, three switches (`RUN_PHASE0_CHECKS`, `RUN_GOLD_CONTROL`, `RUN_AGENT`). It mirrors the
organizers' Getting-Started recipe (wheelhouse install, `VllmServer`, `swegemma.evaluate.Evaluator` with the
subprocess sandbox) and writes everything under `/kaggle/working/runs/<EXP_ID>/` in the layout that
`tools/analyze_traces.py` and `tools/estimate_budget.py` read.

**Attach as inputs** (Add Input): competition `gemma-4-developer-agent`, dataset
`metric/gemma-4-developer-agent-wheelhouse`, model `google/gemma-4/Other/gemma-4-31b-it-qat-w4a16-ct/2`
(or the `.../gemma-4-e4b-it-qat-w4a16-ct/2` / `.../gemma-4-e2b-it-qat-w4a16-ct/2` input when the zip's agent.yaml names a
small variant; the vLLM cell takes the model from the zip).
**Accelerator**: NVIDIA L4 x4 (competition-attached notebooks only). **Internet**: off.
The submission zip comes either from a private Kaggle dataset you upload (`SUBMISSION_ZIP`) or embedded as
base64 by `python tools/build_notebook.py --embed dist/submission.zip`.

**Status: UNTESTED on Kaggle.** This file was written without access to kaggle.com, the wheelhouse or the
harness source. Cells marked `# VERIFY` use API names taken from a community mirror of the starter notebook
and may need adjusting against the real `getting-started-gemma-4-developer-agent` notebook. Read the
starter notebook first and diff the server/evaluator cells.

Edit the parameters in the next cell, save a notebook *version* per experiment, then download
`/kaggle/working/runs/<EXP_ID>.zip` and run `python tools/analyze_traces.py runs/<EXP_ID>`.

In [ ]:
# ---------------------------------------------------------------- parameters (edit me)
EXP_ID = "EXP-062-r"   # overridden by --set
RUN_PHASE0_CHECKS = True          # Q1/Q2/Q3/Q7/Q9 answers + HARNESS_README copy + sample_submission dump
RUN_GOLD_CONTROL = False   # overridden by --set
RUN_AGENT = True                  # run our submission on the sound tasks
MEASURE_DECODE_SPEED = True       # Q10: one timed completion against the server

# Task selection: explicit ids win; otherwise the first N_TASKS of the (optionally repo-filtered) dataset.
TASK_IDS = ["fastapi_14301", "fastapi_14448", "fastapi_14605", "httpx_3672", "requests_7205", "requests_7309", "rich_3006", "rich_3468"]   # overridden by --set
N_TASKS = 8   # overridden by --set
REPO_FILTER = ""                  # e.g. "Textualize/rich"; "" = all repos
EXCLUDE_IDS = []                  # known-unsound tasks from a previous gold control (experiments/README.md Q5)

# Where the submission comes from (first that exists wins). Upload dist/submission.zip as a private dataset,
# or embed it: python tools/build_notebook.py --embed dist/submission.zip
SUBMISSION_ZIP = "/kaggle/input/datasets/YOUR_KAGGLE_USERNAME/gemma4-sub-v1/submission.zip"   # VERIFY path
SUBMISSION_B64 = "UEsDBBQAAAAIAAAAIVxWNoSemwEAAN0CAAAKAAAAYWdlbnQueWFtbH2STW8TMRCG7/4Vg3ogleJ8ESK6ByQOlXoACQkOSBWKJvZs1sVf9XhZwq/H620akIDL7kjz8b7zjK/g9stHudqtYbPa7OR6LVdvGkghZPhEjz35bNC+O5Y/yLfw3ropHrrABMHbEzyEAxiGHKBDryG07RjnjqbJrzfweQPRRLLGE8x8GID7wx7HObw8JxYndPZ6DhwgBs7S+O9BYTbBgwouoqphhwzoLzaqTxc0WZid1f4o/4lJzyFRS4m8ouUluXD6eiGu4EPtHp0zulisHAETVf+JUFN6yTA7knMot5K2B2myfMQshy2ud1LleRH0rTny8tw/rbIoFIJluP860qEfJaVMfrL56qYZ9+i9JmWLnK7FYA1n0AkHhgGTL7O4OnN4Ak0t9jYXzx4dNcADTQxF/e7LHObmGY3QxCqZOK7awF2ZwnWnjPztfJ9xv/mo1+YqgzGWez5fRFSwDfxneWE859SrSeWF8cr2miCmgrkcdzxP4SwqiaagEMUaJcy0L9BytV3h/db7V5ri8mIaASCfqvYRc9f88zmJX1BLAwQUAAAACAAAACFc2PyhGNUAAAAYAQAAFQAAAGNvbmZpZ3Mvc2FtcGxpbmcueWFtbD2OMU/EMAyF9/wKSywgtVXauys0GwMbSIwwVbnGaaK2TpQ6Av49oScx2e/z89O7g5eP91peOuhk19dtW8tHBezwxk8n2PUWV08zWL9iBZkmp2lGU8HudEID15/Dn1AbTKDJHFLH8lX0faD1ZjBJW8a0g/HWYqrAU8nFMv7OPkH4Ivh8fnt9aATjFjFpzgkVtI0UHOIYFchmuBz7oqA/i01/jyFzzDxyWJB2BU/t0Al2npbSeZwCWT8rAfCPrtnMyArOcugL9zSt2eDILuTZcUnglFH8AlBLAwQUAAAACAAAACFc6A3n1sQBAAAZAwAAEAAAAGV2YWxfY29uZmlnLnlhbWxdUkuP2kAMvvMrLO1hF4mgJGxKCFIvqz23hx56QyYxMGpmJhpPYPff154Q+rgl39j+HvYTvP/8nuVVCWVefsmKIsu3DVj8OETv+0OLfc9Q1pB9hdcK0HXT2xgcQ50rXJT5HqKx5Md4YGq966TjNf9TLG8Ha9wYSXoAA02cm80zrxdP8KYcDcQLQSDsKDwz3HzgCJaQx0AdDMgMhqHYQOtHFwWalL2kSUImxNF4B7t1vUzMOg6HoTcUwBGJJjyKQtjAIEgX8BThhToTV9BeqP21gtF1fgnsoajg5AOczJWmQoahH4Udopd+6wPtNY2e8CqWgvc2NRx9vNyVt8jyghG8a+muclMu1e4Pza4BulL4lFkd9frtpPR0B/Gsv8moOOQbnclaFECMf8T18NmU221W7nbLNXxz9EhnMlirSrEwZQlpVSuoJiczkCKqtLLMHzHNpZUKTz72ul3xKTpQBnLEEOd1hNFpv5E3kmHGcRRGdXEk487QihZruuw+faXJ6pBMG/nib6LZpTXFgI+QZJUBDQuJXFeSl6dstY7RphNh7yRJumI/oi69WcD/B9joBQr87yE3srUZ/OsoG6hndNqNmF78BlBLAwQUAAAACAAAACFcat9W8yQHAABvDwAAEgAAAHByb21wdHMvYXBwbGllci5tZHVXwZLbNhK96yu6kkOyKUoTz4wP8SU18djZ1KaclOOqVE5DiGxKKJEAFwCt0W7tv+/rBkjKXu9lhgJAdPfrfq+bf/mJTGAyjvh55JDITMk7P/gpUvRdOssuu4N1zGFH7zn6/iNTOjJNkQPZGCems01HXYuD6XuOCe9OoWFqjsYd5LjBkkk2dpZjPplM4pb2fDQfrQ+7zeYDVo+2bdnRIZgWl5tx7OX8BZfRwSZqbddR8mSoCxyPFHj00SYfLoigpTC5SDZF8mdHCW7EHf2SyPFHXBZ5vimwid5Zd9jRA+IexnTJN8fG41r6FwePLcKvwM0Sy/URiSCZeCJc7ch2+eLeN6ZXw3Q2EXZa8kG2igsBMBvdR7hff00/Te2Bkwaulw0cozkUbIqzyft+i2t7jTDZgWmvr4mHiERxwk5nkZZzsIlDVEN77uCqXFLlV2Eln/8mklyIv35yCRaQ5LwfzdX1fx4RGiOzsNDbAfAfgW2lB88+nAAgpcCMGsgJa8kI/PidgW3xPGCdrMuOq714jRm1npxP4lhzpDjtYeZpNKk57uitdRY5NlKI3sl/Uy6FQVwZPfXAKSh2OJQsQEI1sRrLCH9gM2iVSGUDhDaYLsXNTws2gnVAIQKjGU0/suOMR+NbzpWlV0g+nW5o2Vf/C7yGoUb0LKjzbFBB6nUX/JCZkG/bERJvYW9K44Rt3/f+HOH3++xHPvVq8++xN+7H/2w2j3IvvcCKWnh6sS7eLou36+Ldsni3Lt4vi/fr4stl8aUsPoDbZt9zjgSYAnp6+8uvb1AHTsOm3359pD3q/aQYGHr35s/8G4w7OAHXlLcL9y+R3v1Gj+8f3n6oJIFKuwoZbE7x+v1KKCN4xUIVANyXbL7Y0euj9zHrT3NkOV7q3LiLckVLe/PIjUXqJAOAMfh2apJF7nB0zV/JLHRMqla2hKWX7PcrmI9HqZt6vKQj3t029NVut/uqziHxM4fGxiIGWejaXDGLHFonmV0NthybYPccNUrxTi6XIFWtbuTvk8Y7Xmh70tUnJ6Tc/rOYXS8roZTqHr0VbiUo17ssNhM0QNS7mKirxdiuVvtLYANNzibdai0UDq/XGfFbFCkQETuZOkI8H1Cemw/mNPeBtVZ0Xxb1DL2o6Laiu4ruK3oJRgvEwhBE/moj2RRdqyXbGnW9Iqe3QahkWahfywOE4QjXpTau6g+bvm+fYgpQpFqTKieWitITjs/ziQpJGi1ypdzsL7Iv/E2iSm61jkrucm7FywxzhFpB05LEEKgztkcqDx4qEow9HEVM9RXHz2m+5XGWuFSQVJXN5NB4c+3ELFDz4a2okR7P9X+7uKNiklsjZEvSrI1C6QBkGxy+W33XVRTyOdfp3G+vWSDqieqv6MQ8YkPYqTbgT0weS+EiYp8zvKOfvYQZOVPqfre5/8yc4pJjk44XMSjoUb+Indn7QozFIRHjwqIK9dj6K0/0qlomADUg7263qkjCiuCnA7aBxFPjhwFu5yo4+C+kY/Pyi856h0LYc2NEDXxHA7CRmDv7nKYghLWD8oxD8CHzd0SW+Nmi8eDgHvk6oXlLeYK30l618XtEIZPGPz7BtvoCsNWV01fgCg3vSjdG8eW6GQXUdrOVUERG4M0gKlhIWKoDrRzTj+zn85VuXEpVBdBd9lRW5jfMwYADGe+Vl7tsaTG/VJ9So5ZSLQSW+pubZOCDXR61pYmUtTpBOLDolAX6mv++TFI6wIlfrQcxzrPwLfPL678/vPs5tyPtLeLg74E72Lh2+1NlGKaYCnlnXs0agKy5pp9ayYZ1GEExgcF36TdlgL2ebUdTvENN9DywCq+OCw460FxzHryCRckPnrGFh8kF7nX4lWYhnmfBbhBd4rn9Ua3i/x06gSj3d086NuoPtDotKmGJBJx7x6rvOwxOReHhUTe/uHhlHYYe5G1E70XFKngPaJ/oTQqO9KgxoRJZ5by+ScP4f3g2Nxudf5YLBFl5FYVc38ioCMQaScdb3ei5S7PSXk3wYJ8fpFTBMp8ZqiO3DlkgqQ6ZwiWlBDTnDx0X4fxfOnwbpMZk2mim9zKDaIGKH9ezJcDRGfQsdbgqwWczezzZcWRpJgtyOUzc2ntVhzgBBrjuQ54/RGYEXZFuvzaPPKoL84RYol5mnVVkGEqaep2fJ9fp3Cu/ytSeZIjFKZF65Weh44KQ1B2Wl4lJ+mTuDJ9rZp25afIZtZAp/Ak8n1ip8THwJF8kU6xLI0ESpeDBgh++F6nyrhXkBrwEVKovXmkHUN8iyv6SE6iFAxOvvTKP1/FGZyjEa3rh+2X+DtvRGx3P9FMJn6OtxFu+zQYPIGyjpC0TKXRvm7/t8vcETP3BJkjnnlKU6fC6Nunb+maKQUh0g7LgPuqjH1P9N6Sf8U0AVWjka1S/l4trhUrc6vVyqHxawY198EaGjLH3IavJ5r9QSwMEFAAAAAgAAAAhXMNWJHprAgAAgAQAABQAAABwcm9tcHRzL2RyYWZ0ZXJfMS5tZF1TTU/cMBC9+1fMrZclEq16WSEqyi6wElokVAlxNMnsxsKxU9shRFX/e9/YCYt6ycfM2PPmvTfPfiAdmMZgEgc6J3+g72Rcwz3j4dKciRU9+yGQdkfLa0otU+y0tRwT9T5G82KZ6hZprug6v0lHOvBI1jiO8jMaa6nxK9IpX8DvusZpq2vO/4F1w+FLpOu7q/3tNh/E5calSEmOuQbvoW7J+dQadyS2kSulgIxa/caIo8DbmEu7AdhQSTVwIjJV9JOtH8kgf2oWuPchkXcIRn9Io9BhYhzmMfPnKn8ejOUZxgf8eghBaEr8noS8MsjR4L7kF0aU+oMx3Y+/Sj0Jnej2cR7XmUYjxo0RXkCOBpa+twaCZM56HZEHD15SpQ5j0uDK/Q0aPA7QYq3O6OF+IxPWvjfcCICANrjp4MPnv+A7etze7h72KzRJLRmwbLLkOgF9RbtUmKq9S+Y4+CHSi/X1q0z5TYY7/1q0LTwVmSdIUUBBWJyQQdE4jf6Edi6NHj3J12BQ6uyERw337PkNAI2r7dDARMUGmRfAET2Nm5FXGHe/fZK4TJ3HyBcJHsfcFALQM67kVHat7vjznBTTZMWzvp9IOk+IBDHXh/jQ8/dgAiBnyewkfTc+YyliYAuwIHNIN+jqu47Ft/rFD8XtixXOaLe4JBuv1TLVPJFwFfUU6WZ3v12DslfnR7cqS1jwG9nAQsr+gTaPVze/Fks62u431eKxqexrHDObC3bUgS6Yocun/tuk0lb69BpkAtppMekoK2aSAtdrdXFxoSTn7aynury8VFDjlHLL8ufU093zOtshghiG0gpg1T9QSwMEFAAAAAgAAAAhXMehDrSkAgAA8AQAABQAAABwcm9tcHRzL2RyYWZ0ZXJfMi5tZE1Uy27bMBC86yv21otjoCl6MYIUbew8gMABggJBjrS0sojQXJVLRhGK/nuHlBz7YoivndmZWb9KIhOYhmAjB7okaek7Wd9wz/jxcT7RJb1KCmT83vGKYsdUm6S8pFtcxoOyFXhvxdPQMUrmjSGI39O7cYlJwrzccWfeLZZWqbVBI/VBmlRzs0D9huoOKEw25hKBF8Tv7Mm2M4RpOHxRurn/ub3bkLOeqWGtg92xkkHFD2pTyE+pkcFrxJPDkh7ac4rayaDkhdgEZ3FV0V0NrBk8X+2dqfkMlLw5sC6rCkoQWuD8Poo4LbQPCZ14iRDGOeyMS/rFTobcpjnxDtxLiAQOBqBtHLL8VjXNspbPRflsreNJkrziD1OjeAoh2xL5I2azznqKMrMHxb8g73/8q6qXbB/QPt+jnG0M9rgpCpsIBDJ9X2QYLLj3RnFuPQriaLqHNin5qX4DgOfkWFfVBT09rnOHtfSWi3cBMKjUwuCzVZADPW/uHp62C4DEDvZqyZmPJoI9DIqTUrX4aPdJktLOSf2Wu/yWm/t6WdzWSafySSOsmC3rHV7kRgEcBzmxna+q5EhJDQXzPTfip0aAt4gXsuhrlxoGfklU0cVq8RPhnpgv0e5285L3c9eljVIo8/HMDR/Dq4vjSCgyc94naRwdUG+kH3Oww4idYDEWn+bDzz/JBlAulrkx466lcJnMYI0YyHnLNECVwwEIkG8nKU7jOUfh4pT8ErzOlNxPHWWt1IxKtw+PmxUke/MYmcU09BN/myd+EmX7ROvnn7e/j5H0tNmul8eMjdP/gw5FzSN33MtjLuFQXoFwl5tlpzmmE2wZNgMxQe1s3PZ5xGysoPWqurq6qvKZuNnP6vr6uoIbpyPPw9nRy/3rqsRBIQzD6Qpkq/9QSwMEFAAAAAgAAAAhXBwxahyCAgAApAQAABQAAABwcm9tcHRzL2RyYWZ0ZXJfMy5tZE1UwW7bMAy9+yt42yU1sBW7BEWHbUm3AEUKFAOKHlWbsYUqoifJdY1h/75HyVlySKBQIvne42OeZSQTmKZgEwe6JjnQZ7K+5YHx5dNyE2t6ljGQ8Z3jNaWeqRtNaGva6St7mHNMPCN5GBNJoJhM4hy2MY5M0cyROuGImuK7FYq1ZFp86CUY3/Sa1PTcvNIBJ343TXJa1yRqTOQVOTZv1nelPXsOxtFgUk8jsgGN25L5xmEmwatQ0NRVBfTUmzcmL5REXMzdj2NMiGh95xCZa/rGTiYgBqrApuXwIeIwSAAnj2CUQ5pUskxqfea3yseDdVyY6a/MgZoxBJUy8XtSgfUmcGdRLwkV5ID4Z3DGf/lbVU8qedZyyUc526qY3NpUBDHAMgzOguJkncoQk2qPgrgq70DzrAwaPI6O47q6oof7jTJsZLCQDA8C2qDSIQ/g/68gR3rc/tg97FdoApltitkbHpMFegw/FaUagQe6UcZIL04wQLC8VnIfP5GzHl0z6XykGaMooGhwyFCiaJwmuZhjeRoFPUkaKKjv4AbxDde01xEDSuPGltFfXxddAEfnaf2CvAbd/fZJ48o608iFFI9nbosA6BlXmqXhaI58yRNOnh26fpdhXswVUzgZsZg78O/RBkBebKt9N5KxlGFwTFiiJaSub+R4RAfI9yJjyqVOVrii3ckl2Xi9UVYLo7xaukp3u/vtGpK9epn8qixqwW91S4so+wfaPH69+3WypKftflOfPDaXnY5TVvNi5SAXzHDMWQDcK1l2UW1a2mqfvHqAVpDqrlCnK2ZTBa3X1c3NTZX/Fdwyz+r29rbCNM5XnqeLq6efz+tshwhhGJOuALb6B1BLAwQUAAAACAAAACFcmYPOkKACAADmBAAAFAAAAHByb21wdHMvZHJhZnRlcl80Lm1kZVRNaxsxEL3vr5hbL85Cvy4mpLSx0xqCAyEQcpR3x7sismYrabNZSv9730h2HOjFaGckvY958pOMZALTFGziQF9I9vSVrG95YPz4dOzEmp5kDGR853hJqWdqxKdgmrSgSUJLewl5UdODeWbiFw4zxfmwE7egaDtv0hh4QQeO0XRYxIRKxDUt49qWAmODpxfjRgaASRnFxojPwL9HGzgu8s6DAhQKODsEaccGiK8g42Y0JKLVGyWXmV/33DwTm6Y/XdQCPVjfkemM9THRrOK260dy1nOkHUMOaxWAceJQVxX0U29emLxQEnGxcBlx2kuixjiHylzTD3YygTgZwJmWw4eIxSAhkXgUo+zTpKZnbcuzzEVe7q3jolO/sipqxhB0GIlfk45IO4E7i/uSqFbfMSj+GZzx3/5W1aMODWhv53GdbQ1q3NpU3DXgMgzOYuyTBffBRPStx4VolX2QSaMv97cAuB8dx2V1QXe3K1XYyGBh5pvZOQXvvoIc6H79c3O3RUps6smmmNPlMXywr2mTilOaJtuNgkTsnGBcUPlZxX38VGZSfCrj0bkUUjQ4nFChAE6TnNket0YBJkkDB3Uf8iG+4Zq2GlBQadyo+cu7iy+go/O0/si8hlxNBuqqOsvIFykfz9wWA4CJeOKUlqM58HudiNvsgHotw3x6GiWA/2f8FGTFXUnmUobBMSHMx5JpgSqHAxBg307G8lxOUbigzSklOXi9UVVHRepVNHOkm83tegnLnr1MflGeeuFv9Z0XU7Z3tLr/fvNwiqSn9XZVnzI2l38FfSMq/vwIYRfCcMinQLhXseyixrTAKs5gYCaoFab6VqjTJ2ZTBa+X1eXlZaU9ccd5VldXVxWmcW55nt61Hn89LXMcIoxhTLoC2eofUEsDBBQAAAAIAAAAIVwAY53spwIAAAMFAAAUAAAAcHJvbXB0cy9kcmFmdGVyXzUubWRtVE1r3DAQvetXzK2XjaEtvSwhpelu2oV0A6EQctTas2sRWXIlOY4p/e99I9nZFHoxtkaa9zFPfvQD6cA0BpM40CfyRzyMa7hnPFyaK7GiRz8E0u5keU2aGnM8cpANsdU9V3QjZZ9adJmP0GispU4/MWGZ/OHZ+CFS3aIJDvyQgp4/sUMnOnGKeW/UHdOBW40jgQ7TP4DBD0k4uIIWk07cZSYpDHUaAq9ei8fB1cl4B0m5ceATvlaErplT3/sIsnQI2tUtlhsOFe0SdUOUhqLAdL0tAKOQlHMmxkGa/RpM4FgpBXMIbJmcp+S9jWDQlCZgQrVGH+2miq7Z+hHnoSiwBtq7iJfeh0RgqSn6YxplIhlifUZb5dejsVldk7/4RddoPoTsS+KXJPM76wSVxW6lfvdWu89/lHqQ8QDt9TzamQYmEjcmlUlocOl7a2Sa4kGvI+rGoSFKZR9kEuzN/RsA3A+W41pd0N3tRhTWvjfcCIEAGBkGXH/zFXxH99tvu7v9CiCpJYPpm5w6LTPLc8hO1d4lcxokPQfr6ydR+VHEvf9A1jigZtH5lSaMYg5Vb3FChMq4R39mO2+NHpjkazgo++yER41o7vkZBI2r7dBIRmV38QV0ZJ5IU2FeQe5++yDrojrLyI2Ej2NuigHAjKslgzncb3QiZpMF6lffTyTIkyTZuNN/olZGZifB3fjMpQyDY8IdnZd0A1TfSWZh3wH3JbdaonBBuyUlOXitFlWzIvEq6inSze52u4ZlT86PuDH5Uhf+Rn4CxZT9HW3uv9z8XCLpaLvfVEvGpvLLiGN2c+GOfbALYejyKbmpIpZtlJgWWMHpNczUcWYqd4VOcsVMUvB6rS4vL1W+xHaep7q6ulKYxrnkeHxTevj+uM5xiDCGMWkFsuovUEsDBBQAAAAIAAAAIVx9jjqO+gQAADwKAAARAAAAcHJvbXB0cy9yZWFkZXIubWR1VsFu3DYQve9XDJJD2kKrOi7Qg2+G4yRGAzuIHRQ5WVyJqyVMkQpJ7XqBfnzfDCmt06AXW+IOOTNv3nvUNz+RCprSTtPWhJgoJtVr8ltSlLQaavqGkKBVJzFBjz6a5MORlOvoEEzSiOTlkGq61c+pwkF7nX8KkbRqd9QFtU3knSb9rNpEujOJtsEPdPRTKNsrORJZHB5IjaM1OlAKRkfJLYfEmu6cPfI+ahHmR4RvjdWxoujpsFMp/+bHIxmX/FJ2SGQiKWtlxSpUl5uNdDBY1Xte0NI7R7S+0/Vqxe0P6kjOJ2p3ygEc5Y5pZ1xf09eoqWFwHrmEhpDuVND88+QeWz8MaK4BBKh964Mgupa3Mfg+qCFeUNMDlR610to1FTU28t+ou/K+wx7+f2gbhprrzQPAeYPfy/9OWy0j4RIqchIVgHfQwN1PaZxSxmUOYdBL2OQw85giHxTbYEbAvVq9fk1va7ryLgXMrqKNRgOCAvr1FmOwdvVlZoiJcdJy6JNGJ4rijqG3BtwqyGYOxOOw8ZbHZnqn0hR4hDoEzo2hux6vHx8ePvOQ0hRlHjGXGzTCHe2VnTA9gBb09wkdRi4Tc0eSgYfF9aIvxnt5q+kBJQDxTvOqbp+YXR6TKllzefaYOz+viVuLopKSpmNShKIXjAtDHCY8bhj3BZJKKJOgiJouZwbyvBIThLn/Ai4LbVTgISrBfNQPpOLDQJyiUauF304N2ABCH4J3/QVAiMjRG8RBFpx5U4CuhNnHLB9wSbS3mYxlQTJ4wv6tMpbpO6jU7hYlQk8eCdkIJAIAv/PS12hV6WLQA+ygXq3pXqsAsXvXamnlB0LTq/vP11c372+uHh/u/rq+fUXrNb35rR6Pb+gfYmrT+vysyaLh1INnNxp1a7amJdNpl/CEoWGaL2nyHyy5kDvWoJzB3XBMgSZXdcIWjaac6O35GTjqQKeRecGQ/9KAeSE98nIjiDTadfn11+yLMnY/Z8v7eToC6ffJJylntpDOiwVuvbX+sJ5GuE0GLNsBme0LH96ZlN0i6LV+ZvJkF8yy4T3Z2JQVZSD5nrUA45TqtWS+LN1F8/wTpSQ4+aTsMtRs83CA2cDK+lznlCLmQM3vBx+e4qha3WSV/FHT3+JEL8xWRRgyMFcZkp3azz/zKFb3yY+SkMdzYNNfohhQ3S0mjAeeEkOdDVgk7Kjna+aYlegAg3LxgNmhq6JgbIRAoP4hm5wX1yaWQSiXQEwyNXRxc3//9fpC7jI2HD2Ab7NhZZEytjNbzs4IGHTwIwiRD53FGSVV8TZs2Xu7Ry+8yFfSHDo7yc+m8/7mE8rgmFElKDFoXFXcabnKFuiruTrmVAQMbbEHBBacVp9ubq/vL16wiutYuiY3DRu+o8tBeTIVT8xkRr/kDHCZbKpXX64/3NzdXqxOTt5OITBa7HXzWdvJtYklB6JurG+fTmVVjIRB7//j0CdB/5z+JNg/i15n5hlhHGyJS+dCROuiD+l1g8enWOUVkWZcXM4H0wuFjGObUUlgYOYXHHRs1YiKhUtCR14Vq8918S13wGV59fHy9gPmxzNhiR48xOPg+C3SRXXMZC/7xYmF13AG+Zza6aB/pNQM41zqJijX7urV9e07cHZbvnUmK/Q+WRGMq9dpmWiXddzjulwAazLTJvfk/MFlgxMqlXr5dlg+p7ID6K7K32fIXq/+BVBLAwQUAAAACAAAACFcbmTzR8sAAABYAQAADwAAAHByb21wdHMvcm9vdC5tZGVPQW7DMAy75xV8QJAP7A+7Dj0FXqLWAmypsOV0+X1tZdhlJ0kUJZI3bQUq6YSpJnCFlSD1TmU1XcODxBbctCGGgyCKOydybp3HuGnOQfYLweg6WNt3ZnPsAxYJ/gcSMu148pMSCyFq2utY52X6isHooIJMtXY2zi5ZaCM+aAbLltrO8kDoYGbZO9OGuBhLI6iPl+w8bn8zhc1YxVN1dkjpfzq82OJlcB0GUWk4/7O5TJ9urJ+9emGB0U8X8ai+8b9BTmjPUjz1Mr0BUEsDBBQAAAAIAAAAIVx5tw9abwIAAHMEAAAXAAAAc3ViX2FnZW50cy9hcHBsaWVyLnlhbWxtVMGO0zAQvecrBu2BrdRk29AtbDggJDggJMRhD9wiN540Vh07eCaU1Yp/Z2yny4K4tJ7J+M28Ny+5go/fvpab2xrqTb0vt9ty87oBHjDnX70C6/0E6oiO4ZrUiMDeW1pDOsvPZI07Qm8sruBseAAFU/DjxIKiGDoVgkFKkAGVxvCS5DD5wKCcTvne/EDQQfUc64KfjwM8Tla5d79SzWN61n5ZQkZrCYw08MDhIUKMYBz4IOhVcSXnzs4a2847lrkJNPZqtkLAeQbnHa4yRzXJ9BhgnImBEFOSFZ1gRCIhDdcXjGp6aG73d+Xtm42M6qc0wOykYwas4BODyTytinAc78d5B281Ac2H0XA7Ke4GUc+LTjkF6PSiz+zgOkndytEJGWl6V2/Ku7peJSiZf4hyByylIt/iIIOLPjKJ4bcCy/gzgcbCs5+tKKZOmZubtQwlMwz/abSvy+2m3q3WcB5MN6QLyyp7Hw5GUxT3XrLvP3yWLrJ6FbtSp5ZRjCMOc8fGyzZcJ/I9y1A7s7FJyK10SXy0F2vEpQihCJNQRBZiwzOjzq5IjJJmh6A6uSGoRgvp5bEK0Yk9VoUTUzaXtRYaqQtmis0buH+yYbok2KgN05Nxsnljro1mXkM3YHciQCVSRDJrOCFOtDg20GLw3IwSnUkRobwb8ZyXm8vT0qti9BptA0ccR1XuStwdSsPld8Xleae2+7Lj4pleDbxYjAxVdZMXQTcLt2rUxb82by4+L9Ir2hQAZfSUFEhDp3Ms72AimKInuik6B8P4Jzwit+JinimFzw1ciHcwyPYvzeN/b45/z5xzdHP5SlQParTFb1BLAwQUAAAACAAAACFcoxFEsaMBAABOAwAAFQAAAHN1Yl9hZ2VudHMvZHJhZnQueWFtbI2SQW/TQBCF7/kVT+qBRLKNkyatcIUQh3JBQj1yswZ7HK+02bV21g0I8d+ZXYdSCQ65rOzZ55n3vvENHr8+lfVhh129uyu327K+bzCYZ0YfaIgcBMZhokDWssX6M97jUOBs4qj1ODL2iB538EN+641MFLtxU+GRuhEjaYMo8GeHIdDJuCMoZukQvIvpu3RvnMQwd9F4V0A8nMcUeDDWwghkpMB9AXL9S7fIp4kDxTkw1nV1W6CuDum4T8e7AtuqfkA3emFt6bwOnR33m2p1s1g7BxNZezkGf6cugnsTUxhhEfUBiRQZs+s5LDTaLw9LyD9ohDkXApOK3og+TD5ErFPx52TJffiVjVrttMn21ciYILAV9W1cZ+ee205RsNNkTu1oTslt8272NZ59mqbOHZ24Wcav6KgftJ0lkQZPlwV9TMVVz9IFMyWYDT6lZRoNMbEeLiX0trSacSEQpABnHrrTF7rkjpaz4VecL8I8/x9wSXtB+h+M1Urmb232LM0KKKGRB3Ns9W8ZG/y9fHuB226rH3SyV0l310tvr5fur5ceFulvUEsDBBQAAAAIAAAAIVwJvygZxgEAACUDAAAZAAAAc3ViX2FnZW50cy9kcmFmdGVyXzEueWFtbIVSwY7TMBC95yse2sOC1GSTbNsVviAkel1xWySEIjeZJlYdO9gTWoT4d2xvWlbLgVNmJjNv3vi9G+y+fM7LTY26rLd5VeXlg0Dn5IHJoYI9YLOCNL0m+FFqTZ7RDqFAKzCNEznJsyOUxX0BtlZ7fP0WBjoo0+q5o6a1hsmwh7GGBBTDE3nwQHAkO3K3PgSTdRxqzs79gF+TlubD7wRjLA/K9CDtqchuMMpzY2eeZm7YHsl43JcPdWqNjcfQ2+znridGVdZrvOUhYob9dE6EOhycHdP+dOc7AfkcQfkQarknjUnPHlZ3zxzoFI498yplEu3My4S2PtyiAvrj7gl7bdvjCt4m9FZOETEckNIfNjypv0W92Rb4uMyfFA+hA20AilfuHj9BKxOG7RzfTPq4rMiMHOmqS1NlHfnWqYmVNQJPTr1U6786ZaPtSAv0NI4yX+e03ueK8++S89NaVtu85UwZz25unxe8WbREUdxN4fEm9ndXLsXYZa+1FknsbNHpSD8X7oF58ogIJsl6MpHVdSp+D6oXGV4yFpFyLNmpmWLyfrNkR4HtOsT/OEIkS8SuiyH+IuO1S0SySfpzueLiGIGDDK7L/gBQSwMEFAAAAAgAAAAhXObOJcEdAQAA4QEAABkAAABzdWJfYWdlbnRzL2RyYWZ0ZXJfMi55YW1sdY/LTsMwEEX3/opBbOs0CWmres8eiQVICEVuPEmtxA/ssQp/T1zaKBtW9rzuPfcRnt9feLmroS7rPa8qXh4EqCB7wgA1uB52G5B2mBA6mSJugNB4DJJSQCiLXQGv0iCQc1PcwEVb5S7zgZrXfQQZF7GqYHbeXNTbmimMXdCetLMC3oJee/7jxoxTOAkY0BjJG47NiWviX5L4pZHVnnfEtI0UUvcn+6BtNyWFUBRbH5zxFLcLQWEUu83bzllCS1GAdRaZS+QTtSP+3Ihn3mtIAR+fbECbqZar/PZ6EAzWxCIj55bzrc/F8V6NAvbN/Dfyu705kRvRzupP5aHOW2dtR22HlfKqeUpqQBJQlXVzndxT0Nml4ZxT9HKKyH4BUEsDBBQAAAAIAAAAIVwXAE/THQEAAOEBAAAZAAAAc3ViX2FnZW50cy9kcmFmdGVyXzMueWFtbHWPy07DMBBF9/6KQWzrNEnTVnjPHokFSAhFJp64VuIH9liFvycpIcqGlT2ve8+9h8fXJ14ea6jL+sSripdnASrKnjDCAXwPxx1Ip0cEnWVUOyC0AaOkHBHK4lzAs7QI5P2YdnA1TvnrdKCgkyGBTKtYVTA3ba7q7YEpTF00gYx3Al6i2Xr+48asVzgK0Git5A3H5oMb4p+S+LWR1Yl3xIxLFHP3K3tnXDdmhVAU+xC9DZT2K0FhFVvmbecdoaMkwHmHzGcKmdoBvxfiifcWUsDbO9PoZqr1an57owWDLbGYkeeWD22Yi4fjUg0CTs30t/KrXZzID+gm9UN5rueti3GDcXqjvGl+ZKWRBFRl3dwmfyno4rO+zCl6OSZkP1BLAwQUAAAACAAAACFcwTtzQiUBAAADAgAAGQAAAHN1Yl9hZ2VudHMvZHJhZnRlcl80LnlhbWyNj0tPhDAUhff9Fde4nTKAzEzs3r2JC02MIR166TTQh+0l6L8XEAlLV72vnvOde3h6e+b5qYQyL8+8KHh+EaCibAkjVOBbOB1AOt0jNN5RlA3B6KOC1selOAChDRglDREhzx4zeJEWgbzv0wFG45QfJwUFjQwJZNrUi4y56XKzqyumMDXRBDLeCXiNZg/xX3tmvcJegEZrJa84VlduiH9K4mMlizNviBmXKA7Nr8+dcU0/KIQsO4bobaB03JAyq9i6r2cCdJQEOO+Q+YHCQHWH32uEKcCSWsD7B9PoZqrt1/y2RgsGe2IxI88jH+qwNKe16wScq6m28qtench36Cb1h/xSzlc34zrj9E55N7wOSiMJKPKyWjZ/KejmB32bU7SyT8h+AFBLAwQUAAAACAAAACFcfKwpticBAAD5AQAAGQAAAHN1Yl9hZ2VudHMvZHJhZnRlcl81LnlhbWyFj8tOwzAQRff+ikFs6zQJSat6zx6JBUgIRW48cazED+yJCn9PUkJUVqw8nsc9997D4+sTz+sSyrw88KLg+VGAirIjjFCD76DegXR6RJCgTNdhREeQehlwB4Q2YJQ0RYQiyzN4lhaBvB/TDi7GKX+ZjxW0MiSQaRMuMubmzY3U1ExhaqMJZLwT8BLNH/7/ZGa9wlGARmslrzhWZ26If0jil0oWB94SMy5RnNofxJ1x7TgphCzbh+htoLTf3GRWsXXetN7RDE4CnHfI/ERhombAr9X97P0aWMDbO9PoFlfb1fJ2RgsGt47FYnlp+dAEAXl2qtffIOBQzbWVn81KIj+gm9Uf8mO5bPXGDcbpG+Wb5nlSGmnWz8vqOvlNQb2fdL+k6OSYkH0DUEsDBBQAAAAIAAAAIVw1DA7nVQEAAF8CAAAYAAAAc3ViX2FnZW50cy9waXBlbGluZS55YW1shZE/T8MwEMX3fIonsYDUlLYSCGVAYmBjQMDAVh3OpbFw7OC7tuTbYzsFNtjs+/Pez89nuH99rFfXa2xWm+t6va5XNw2057l+tcHLBjEExRAO3KLnyNh705PfpSt/Gh4VVgWeBsa5HHlLO/aK+hajHdlZzxcLSAB5PLjhrjSLokmVpNMidB00JJUlXpKzCb6zO4ykvUCUJkR2pPbAeSqzaWQuGssq2zY/TlXx3hpHIg2e+WOfrpZcca1aFhPtqDb4Bk9MbRGLPAaxGuKEo9UelCrUckxNUvTBtVIqyTw4QfBuwtsE60Xj3mSxBRwruhkwuNqxCI7RKkcBk+nRRuo0rXJKjIyCW5sWYhhKdJkg6iLTeNA4JoMMVpYkGSHEzFPoskYesd+ApmfzfrLJYcr+bUiiyyod5q+QpgLqU6zbHGuD3+bl/NrlRIP7c67g/D92gpsHvwBQSwMEFAAAAAgAAAAhXOArw+M3AgAA9wMAABYAAABzdWJfYWdlbnRzL3JlYWRlci55YW1sbVM9c9swDN31K9BkaHJnKXbiOKk6dGrXduzmYyRI4pkiGQKMrOv1vxek7CTtdZIIkg/vA7yErz9/lOv7W7hd3+7KzaZcP9QQULXAzhkCZ80MTzNoSxxiw9rZCkK0+8aNo7ItaAIFXUAEGtAYuKIJe5Q9wCM2MV/wc719uF4BObiwDrDVvMBfwOBMS6kBDwg+uNEzJFjr+FxNJ8Fo4s95mchp2xM0AzaH1xKGj1I6kbJqRKqKS6HdmNiisLWMlgla7FQ0XL+7lyR0OhBngloOTdq2bkr13F/RAQSPVI+AWkoBJjWvMk91RjxfOiD6/9zT3VJLRgkuvghIwFKchKscwXYN3zYbMI5YOEC0iZl1Fq+TDhfZR94fcAZvlF3od9oqA4xHhiseXOwHBq+CCMBjlt2uwJhxL/0tpxAe7z+Vj7u76yRc3FtoTkEzBspq0lp5b7T05iEkyFz7lZp++S29RvljXME06GZISgI+RS3yO8kOiGVTZibrm2XvOSItgYq1FMOLfsEck1fLLBUpqvqURNEiNUH7tFPDd492odi4VqwbFMMYBa4ZlBVLE2pA75LgdEoTRWG2OGNOf3iURtDEEMQDOd4L9KtUnYZCbDohVsUojUwNeX7LbYnbp1Jz+ay4nLZqsysbLt49hBo+nOYLqupmGV66WaRUY1v8O3z1eVaKtzjrnGeRX0NdAJTZin0SsKzenlpe98j75HKkQlLFIH6f8dO30/3frJYa3ZCS5OTZVLMaTfEHUEsBAhQAFAAAAAgAAAAhXFY2hJ6bAQAA3QIAAAoAAAAAAAAAAAAAAKQBAAAAAGFnZW50LnlhbWxQSwECFAAUAAAACAAAACFc2PyhGNUAAAAYAQAAFQAAAAAAAAAAAAAApAHDAQAAY29uZmlncy9zYW1wbGluZy55YW1sUEsBAhQAFAAAAAgAAAAhXOgN59bEAQAAGQMAABAAAAAAAAAAAAAAAKQBywIAAGV2YWxfY29uZmlnLnlhbWxQSwECFAAUAAAACAAAACFcat9W8yQHAABvDwAAEgAAAAAAAAAAAAAApAG9BAAAcHJvbXB0cy9hcHBsaWVyLm1kUEsBAhQAFAAAAAgAAAAhXMNWJHprAgAAgAQAABQAAAAAAAAAAAAAAKQBEQwAAHByb21wdHMvZHJhZnRlcl8xLm1kUEsBAhQAFAAAAAgAAAAhXMehDrSkAgAA8AQAABQAAAAAAAAAAAAAAKQBrg4AAHByb21wdHMvZHJhZnRlcl8yLm1kUEsBAhQAFAAAAAgAAAAhXBwxahyCAgAApAQAABQAAAAAAAAAAAAAAKQBhBEAAHByb21wdHMvZHJhZnRlcl8zLm1kUEsBAhQAFAAAAAgAAAAhXJmDzpCgAgAA5gQAABQAAAAAAAAAAAAAAKQBOBQAAHByb21wdHMvZHJhZnRlcl80Lm1kUEsBAhQAFAAAAAgAAAAhXABjneynAgAAAwUAABQAAAAAAAAAAAAAAKQBChcAAHByb21wdHMvZHJhZnRlcl81Lm1kUEsBAhQAFAAAAAgAAAAhXH2OOo76BAAAPAoAABEAAAAAAAAAAAAAAKQB4xkAAHByb21wdHMvcmVhZGVyLm1kUEsBAhQAFAAAAAgAAAAhXG5k80fLAAAAWAEAAA8AAAAAAAAAAAAAAKQBDB8AAHByb21wdHMvcm9vdC5tZFBLAQIUABQAAAAIAAAAIVx5tw9abwIAAHMEAAAXAAAAAAAAAAAAAACkAQQgAABzdWJfYWdlbnRzL2FwcGxpZXIueWFtbFBLAQIUABQAAAAIAAAAIVyjEUSxowEAAE4DAAAVAAAAAAAAAAAAAACkAagiAABzdWJfYWdlbnRzL2RyYWZ0LnlhbWxQSwECFAAUAAAACAAAACFcCb8oGcYBAAAlAwAAGQAAAAAAAAAAAAAApAF+JAAAc3ViX2FnZW50cy9kcmFmdGVyXzEueWFtbFBLAQIUABQAAAAIAAAAIVzmziXBHQEAAOEBAAAZAAAAAAAAAAAAAACkAXsmAABzdWJfYWdlbnRzL2RyYWZ0ZXJfMi55YW1sUEsBAhQAFAAAAAgAAAAhXBcAT9MdAQAA4QEAABkAAAAAAAAAAAAAAKQBzycAAHN1Yl9hZ2VudHMvZHJhZnRlcl8zLnlhbWxQSwECFAAUAAAACAAAACFcwTtzQiUBAAADAgAAGQAAAAAAAAAAAAAApAEjKQAAc3ViX2FnZW50cy9kcmFmdGVyXzQueWFtbFBLAQIUABQAAAAIAAAAIVx8rCm2JwEAAPkBAAAZAAAAAAAAAAAAAACkAX8qAABzdWJfYWdlbnRzL2RyYWZ0ZXJfNS55YW1sUEsBAhQAFAAAAAgAAAAhXDUMDudVAQAAXwIAABgAAAAAAAAAAAAAAKQB3SsAAHN1Yl9hZ2VudHMvcGlwZWxpbmUueWFtbFBLAQIUABQAAAAIAAAAIVzgK8PjNwIAAPcDAAAWAAAAAAAAAAAAAACkAWgtAABzdWJfYWdlbnRzL3JlYWRlci55YW1sUEsFBgAAAAAUABQANAUAANMvAAAAAA=="               # filled in by tools/build_notebook.py --embed
# Model: taken from the embedded zip's agent.yaml. Set MODEL_NAME only as a cross-check (the run asserts the
# zip declares exactly this name), e.g. --set MODEL_NAME='"gemma-4-e2b-it-qat-w4a16-ct"'. The kernel-metadata
# model_sources must attach google/gemma-4/other/<name>/2 for the same name.
MODEL_NAME = "gemma-4-e4b-it-qat-w4a16-ct"   # overridden by --set

# Compaction settings: Q7 ANSWERED [V] from HARNESS_README.md section 7.2 (interval 15, overlap 2,
# token_threshold 14,336, event_retention_size 5). The 'interval 5' reading confused retention with interval.
COMPACTION_INTERVAL = 15   # [V] HARNESS_README §7.2: interval 15, overlap 2, threshold 14,336, retention 5
COMPACTION_TOKEN_THRESHOLD = 14336

In [ ]:
# ---------------------------------------------------------------- environment + wheelhouse (hosts' recipe)
# Adapted from the organizers' Getting-Started notebook as mirrored by community repos.  # VERIFY
import glob, importlib, json, os, shutil, subprocess, sys, time, zipfile, io, base64
from pathlib import Path

os.environ.update({
    'LITELLM_LOCAL_MODEL_COST_MAP': 'True', 'TRANSFORMERS_NO_TF': '1', 'VLLM_WORKER_MULTIPROC_METHOD': 'spawn',
    'VLLM_MEMORY_PROFILER_ESTIMATE_CUDAGRAPHS': '1', 'VLLM_ENGINE_READY_TIMEOUT_S': '1200',
    'VLLM_NO_USAGE_STATS': '1', 'OTEL_SDK_DISABLED': 'true', 'PYTORCH_CUDA_ALLOC_CONF': 'expandable_segments:True',
})
WHEELHOUSE_DIR = next(Path(p) for p in ['/kaggle/input/datasets/metric/gemma-4-developer-agent-wheelhouse',
                                        '/kaggle/input/gemma-4-developer-agent-wheelhouse'] if Path(p).exists())
for pat in ('/usr/local/lib/python*/dist-packages/*cutlass*.pth', '/usr/local/lib/python*/site-packages/*cutlass*.pth'):
    for pth in glob.glob(pat):
        try:
            os.unlink(pth)
        except OSError:
            pass
tmp_whl = Path('/tmp/wheelhouse'); tmp_whl.mkdir(parents=True, exist_ok=True)
for w in WHEELHOUSE_DIR.glob('*.whl'):
    if 'cutlass' in w.name.lower():
        continue
    name = w.name.replace('cu128', '+cu128') if ('cu128' in w.name and '+' not in w.name) else w.name
    if not (tmp_whl / name).exists():
        os.symlink(w, tmp_whl / name)
# The wheelhouse ships cp312 binary wheels (orchestrator env = Python 3.12), but the grading sandbox and
# Kaggle's CPU image are Python 3.13 (docker/Dockerfile.sandbox is `python:3.13-slim`). A single bulk
# `pip install ... check=True` aborts the WHOLE install when one wheel is incompatible, which is what killed
# the first EXP-000 push (apache_tvm_ffi cp312 on py3.13). Filter by interpreter tag, then assert that the
# packages we actually need for this run arrived.
# Import names, not distribution names: the `google_adk` wheel installs the `google.adk`
# namespace package, so importing 'google_adk' always fails. [V] google_adk-1.36.1.dist-info
REQUIRED_PKGS = ('swegemma', 'adk_submission', 'adk_eval_core', 'google.adk')

def _wheel_is_compatible(path: Path) -> bool:
    """True when the wheel's tags intersect this interpreter's supported tags."""
    try:
        from packaging.tags import sys_tags
        from packaging.utils import parse_wheel_filename
    except Exception:  # noqa: BLE001 - fall back to accepting everything
        return True
    try:
        _, _, _, tags = parse_wheel_filename(path.name)
    except Exception:  # noqa: BLE001 - odd filename: let pip decide
        return True
    supported = set(sys_tags())
    return any(t in supported for t in tags)

all_wheels = sorted(tmp_whl.glob('*.whl'))
usable = [w for w in all_wheels if _wheel_is_compatible(w)]
skipped = [w.name for w in all_wheels if w not in usable]
T_INSTALL = time.time()
proc = subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '--no-deps', '--force-reinstall',
                       *(str(w) for w in usable)], capture_output=True, text=True)
if proc.returncode != 0:
    print('bulk install failed; retrying wheel-by-wheel to isolate the bad ones')
    failed = []
    for w in usable:
        r = subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '--no-deps', '--force-reinstall', str(w)],
                           capture_output=True, text=True)
        if r.returncode != 0:
            failed.append(w.name)
    skipped += failed
importlib.invalidate_caches()
print(f'wheelhouse: installed {len(all_wheels) - len(skipped)}/{len(all_wheels)} wheels in {time.time() - T_INSTALL:.0f}s '
      f'(python {sys.version_info.major}.{sys.version_info.minor})')
if skipped:
    print(f'  incompatible/failed ({len(skipped)}):', ', '.join(sorted(skipped)[:12]))
missing = []
for _pkg in REQUIRED_PKGS:
    try:
        importlib.import_module(_pkg)
    except Exception as _exc:  # noqa: BLE001
        missing.append(f'{_pkg} ({_exc.__class__.__name__}: {_exc})')
assert not missing, 'required packages unavailable: ' + '; '.join(missing)
print('  required packages present:', ', '.join(REQUIRED_PKGS))

DATA_DIR = next(Path(p) for p in ['/kaggle/input/competitions/gemma-4-developer-agent',
                                  '/kaggle/input/gemma-4-developer-agent'] if Path(p).exists())
RUN_DIR = Path('/kaggle/working/runs') / EXP_ID
for sub in ('traces', 'patches', 'logs', 'phase0'):
    (RUN_DIR / sub).mkdir(parents=True, exist_ok=True)
print('data:', DATA_DIR, ' run dir:', RUN_DIR)

In [ ]:
# ---------------------------------------------------------------- Phase 0 checks (plan §3.4)
# Answers Q1/Q2/Q3/Q7/Q9 by reading the installed harness source; saves HARNESS_README.md and
# sample_submission/ to the run dir so they can be downloaded and read offline.
if RUN_PHASE0_CHECKS:
    import importlib.metadata as md, re
    report = {'python': sys.version, 'packages': {}}
    for pkg in ('swegemma', 'adk-submission', 'adk-eval-core', 'google-adk', 'vllm', 'litellm'):
        try:
            report['packages'][pkg] = md.version(pkg)
        except md.PackageNotFoundError:
            report['packages'][pkg] = None
    for name in ('HARNESS_README.md',):
        src = DATA_DIR / name
        if src.exists():
            shutil.copy(src, RUN_DIR / 'phase0' / name)
    ss = DATA_DIR / 'sample_submission'
    if ss.exists():
        shutil.copytree(ss, RUN_DIR / 'phase0' / 'sample_submission', dirs_exist_ok=True)
        print('--- sample_submission files (Q2) ---')
        for p in sorted(ss.rglob('*')):
            if p.is_file() and p.suffix in ('.yaml', '.yml', '.md', '.txt', '.json'):
                print(f'===== {p.relative_to(ss)}'); print(p.read_text()[:4000])

    def grep_pkg(pkg: str, patterns: list[str]) -> dict:
        hits = {}
        try:
            root = Path(importlib.import_module(pkg).__file__).parent
        except Exception as exc:  # noqa: BLE001
            return {'error': str(exc)}
        for pat in patterns:
            rx = re.compile(pat)
            for f in root.rglob('*.py'):
                for i, line in enumerate(f.read_text(errors='ignore').splitlines(), 1):
                    if rx.search(line):
                        hits.setdefault(pat, []).append(f'{f.relative_to(root)}:{i}: {line.strip()[:160]}')
        return hits

    report['Q1_tool_registry'] = grep_pkg('swegemma', [r'register\(', r'def (run_command|read_file|edit_file|write_file|get_status|submit_patch|get_code_neighbors|search_similar_code|get_code_subgraph)\b'])
    report['Q1_skill_tools'] = grep_pkg('adk_submission', [r'run_skill_script|load_skill_resource|SkillToolset'])
    report['Q2_eval_config_keys'] = grep_pkg('swegemma', [r'max_time_minutes|max_tool_calls|max_turns|timeout_seconds', r'load_submission_eval_config'])
    report['Q2_agent_tool_schema'] = grep_pkg('adk_submission', [r'agent_tool|config_path|skip_summarization'])
    report['Q3_thinking_forwarding'] = grep_pkg('adk_submission', [r'thinking_budget|thinking_token_budget|enable_thinking|include_thoughts'])
    report['Q7_compaction'] = grep_pkg('swegemma', [r'compaction_interval|token_threshold|EventsCompactionConfig'])
    report['Q9_extensions_limits'] = grep_pkg('swegemma', [r'ALLOWED_SUBMISSION_EXTENSIONS|MAX_SUBMISSION_SIZE_BYTES|allowed_file_extensions|max_total_size_bytes'])
    report['Q2_grading_timeout'] = grep_pkg('swegemma', [r'timeout', ]) if False else grep_pkg('swegemma', [r'verification.*timeout|timeout.*verif|command_timeout_seconds'])
    (RUN_DIR / 'phase0' / 'phase0_report.json').write_text(json.dumps(report, indent=1))
    print(json.dumps({k: (v if k == 'packages' else {kk: len(vv) for kk, vv in v.items()} if isinstance(v, dict) else v)
                      for k, v in report.items()}, indent=1))
    print('full grep hits saved to', RUN_DIR / 'phase0' / 'phase0_report.json')

In [ ]:
# ---------------------------------------------------------------- tasks + candidates
from swegemma.models import load_tasks  # [V] swegemma/models/__init__.py exports load_tasks(path)->list[Task]
TASKS_PATH = DATA_DIR / 'tasks.jsonl'
all_tasks = load_tasks(TASKS_PATH)
from collections import Counter
print('repo counts:', Counter(t.repo for t in all_tasks))
if TASK_IDS:
    want = set(TASK_IDS)
    candidates = [t for t in all_tasks if t.instance_id in want]
else:
    pool = [t for t in all_tasks if (not REPO_FILTER or t.repo == REPO_FILTER) and t.instance_id not in set(EXCLUDE_IDS)]
    candidates = sorted(pool, key=lambda t: t.instance_id)[:N_TASKS]
print(f'{len(all_tasks)} tasks; candidates ({len(candidates)}):', [t.instance_id for t in candidates])

In [ ]:
# ---------------------------------------------------------------- unpack our submission
AGENT_DIR = Path('/kaggle/working/agent')
shutil.rmtree(AGENT_DIR, ignore_errors=True); AGENT_DIR.mkdir(parents=True)
if SUBMISSION_B64:
    zipfile.ZipFile(io.BytesIO(base64.b64decode(SUBMISSION_B64))).extractall(AGENT_DIR)
elif Path(SUBMISSION_ZIP).exists():
    zipfile.ZipFile(SUBMISSION_ZIP).extractall(AGENT_DIR)
elif not RUN_AGENT and (DATA_DIR / 'sample_submission').is_dir():
    # A gold-control run never calls a model, so it needs no submission of ours. Fall back to the
    # official sample_submission: it is a real, schema-valid submission dir, so the model/adapter/limits
    # validation below still runs for real instead of being skipped.
    shutil.copytree(DATA_DIR / 'sample_submission', AGENT_DIR, dirs_exist_ok=True)
    print('no submission supplied; using official sample_submission (RUN_AGENT=False)')
else:
    raise SystemExit('no submission: set SUBMISSION_ZIP to your uploaded dataset file or embed with tools/build_notebook.py')
print('agent files:', sorted(str(p.relative_to(AGENT_DIR)) for p in AGENT_DIR.rglob('*') if p.is_file()))
import yaml
EVAL = (yaml.safe_load((AGENT_DIR / 'eval_config.yaml').read_text()) or {}).get('evaluation', {}) \
    if (AGENT_DIR / 'eval_config.yaml').exists() else {}
if not RUN_AGENT:
    # The official sample eval_config is deliberately tiny (1 min / 10 tool calls) to demo the schema.
    # Gold control runs no agent loop, so those caps only risk failing a sound task for budget reasons
    # and corrupting the Q5 answer. Give the grader room; the agent budget is irrelevant here.
    EVAL = {**EVAL, 'timeout_seconds': 300, 'max_time_minutes': 60.0, 'max_tool_calls': 100}
print('eval_config:', EVAL)

In [ ]:
# ---------------------------------------------------------------- vLLM server (hosts' recipe)  # VERIFY
# CPU-safe imports only: a gold-control run (RUN_AGENT=False) never loads vLLM/litellm/torch, and on
# Kaggle's Python 3.13 CPU image those cp312 wheels are not installed at all.
from adk_submission import ModelRegistry, discover_adapters, discover_declared_models
from swegemma.config import ALLOWED_ADAPTER_EXTENSIONS, EvalConfig, build_submission_limits

# D-004 2026-11-05: TARGET was hard-coded to 31B; it now comes from the zip, so a small-variant candidate
# runs without editing this file. Mirror of tools/submission_rules.py ALLOWED_MODEL_NAMES (the notebook runs
# on Kaggle and cannot import tools/). 12B is excluded there and here: it needs vLLM --hf-overrides
# (HARNESS_README §3.1 MODEL_VLLM_EXTRA_ARGS), which this cell does not pass.
ALLOWED_MODELS = ('gemma-4-31b-it-qat-w4a16-ct', 'gemma-4-e4b-it-qat-w4a16-ct', 'gemma-4-e2b-it-qat-w4a16-ct')
# [V] adk_submission.discovery.discover_declared_models(submission_dir) -> set[str]; walks the root
# agent, sub_agents and agent_tools. There is no `validate_single_declared_model` anywhere in the
# wheelhouse -- that was an invented name. Exactly one model may be declared, so assert the set size.
declared = discover_declared_models(str(AGENT_DIR))
assert len(declared) == 1, f'submission must declare exactly one model, got {sorted(declared)}'
TARGET = next(iter(declared)).strip()
for _prefix in ('openai/', 'google/', 'hosted_vllm/', 'custom/'):   # [V] HARNESS_README §3.2 normalize_model_name
    if TARGET.startswith(_prefix):
        TARGET = TARGET[len(_prefix):]
        break
assert TARGET in ALLOWED_MODELS, f'declared model {TARGET!r} not in {ALLOWED_MODELS}'
assert not MODEL_NAME or MODEL_NAME == TARGET, f'MODEL_NAME={MODEL_NAME!r} but the zip declares {TARGET!r}'
MODEL_DIR = f'/kaggle/input/models/google/gemma-4/other/{TARGET}/2'   # [V] path layout EXP-006 resolved 'exact'
print('model:', TARGET, '->', MODEL_DIR)
adapters = discover_adapters(str(AGENT_DIR), adapter_extensions=ALLOWED_ADAPTER_EXTENSIONS)
# EXP-000 gold control needs no model: it applies each task's reference patch and runs test_patch.
# Gate the server so a gold-control-only run costs ZERO GPU quota (plan §5.7) and can use a CPU notebook.
NEED_SERVER = RUN_AGENT or MEASURE_DECODE_SPEED
server = None
SERVER_START_S = None
target_id = None
models = ModelRegistry()

if NEED_SERVER:
    import litellm, torch
    from adk_submission import VllmConfig, VllmServer
    from google.adk.models.lite_llm import LiteLlm
    try:
        from adk_submission.resolvers.generation import install_litellm_reasoning_patch
        install_litellm_reasoning_patch()
    except Exception as exc:  # noqa: BLE001
        print('reasoning patch not installed:', exc)
    litellm.drop_params = True
    gpu_count = torch.cuda.device_count() if torch.cuda.is_available() else 0
    assert gpu_count >= 1, 'no GPU: select the L4 x4 accelerator'
    assert Path(MODEL_DIR, 'config.json').exists(), (
        f'{MODEL_DIR} missing: kernel-metadata.json model_sources must list google/gemma-4/other/{TARGET}/2')
    tp = 4 if gpu_count >= 4 else (2 if gpu_count >= 2 else 1)
    t0 = time.time()
    server = VllmServer(VllmConfig(
        model=MODEL_DIR, port=8000, host='127.0.0.1', tool_call_parser='gemma4', reasoning_parser='gemma4',
        default_chat_template_kwargs={'enable_thinking': True}, max_model_len=32768,
        dtype='bfloat16' if torch.cuda.is_bf16_supported() else 'auto', gpu_memory_utilization=0.90,
        enable_auto_tool_choice=True, enable_lora=True, max_loras=8, max_lora_rank=128,
        tensor_parallel_size=tp, startup_timeout=60 * 20), adapter_manifest=adapters)
    server.start()
    SERVER_START_S = round(time.time() - t0, 1)
    print(f'vLLM up in {SERVER_START_S}s (tp={tp}, gpus={gpu_count}) at {server.base_url}')
    target_id = getattr(server, 'effective_model_path', MODEL_DIR)
    for alias in {next(iter(declared)), TARGET}:
        models.register(alias, LiteLlm(model=f'openai/{target_id}', api_base=server.base_url, api_key='EMPTY'))
else:
    print('vLLM server SKIPPED (RUN_AGENT and MEASURE_DECODE_SPEED are both False) - no GPU required.')

In [ ]:
# ---------------------------------------------------------------- Q10: decode speed probe
DECODE = None
if MEASURE_DECODE_SPEED:
    import urllib.request
    body = json.dumps({'model': target_id, 'messages': [{'role': 'user', 'content': 'Count from 1 to 400 separated by spaces.'}],
                       'max_tokens': 400, 'temperature': 0.0, 'chat_template_kwargs': {'enable_thinking': False}}).encode()
    req = urllib.request.Request(server.base_url.rstrip('/') + '/chat/completions', data=body,
                                 headers={'Content-Type': 'application/json'}, method='POST')
    t0 = time.time()
    with urllib.request.urlopen(req, timeout=600) as resp:
        out = json.loads(resp.read())
    dt = time.time() - t0
    ct = out.get('usage', {}).get('completion_tokens', 0)
    DECODE = {'completion_tokens': ct, 'seconds': round(dt, 2), 'tok_s': round(ct / dt, 1) if dt else None}
    print('decode probe:', DECODE)

In [ ]:
# ---------------------------------------------------------------- evaluator helpers
import asyncio, concurrent.futures
from google.adk.agents.context_cache_config import ContextCacheConfig
from google.adk.apps._configs import EventsCompactionConfig  # [V] google/adk/apps/_configs.py:50
from swegemma.evaluate import Evaluator

def run_sync(fn):
    try:
        loop = asyncio.get_running_loop()
    except RuntimeError:
        loop = None
    if loop is not None and loop.is_running():
        with concurrent.futures.ThreadPoolExecutor(max_workers=1) as ex:
            return ex.submit(lambda: asyncio.run(fn())).result()
    return asyncio.run(fn())

def make_config(results_dir: Path) -> EvalConfig:
    limits, gen = build_submission_limits()
    return EvalConfig(
        tasks_path=TASKS_PATH, snapshots_dir=DATA_DIR / 'snapshots', results_dir=results_dir,
        submission_dir=AGENT_DIR, models=models, sandbox='subprocess',
        timeout_seconds=int(EVAL.get('timeout_seconds', 300)),
        max_time_minutes=float(EVAL.get('max_time_minutes', 60.0)),
        max_tool_calls=int(EVAL.get('max_tool_calls', 100)),
        max_turns=int(EVAL['max_turns']) if EVAL.get('max_turns') is not None else None,
        limits=limits, generation_constraints=gen, adapter_manifest=adapters,
        context_cache_config=ContextCacheConfig(min_tokens=2048, ttl_seconds=1800, cache_intervals=10),
        events_compaction_config=EventsCompactionConfig(compaction_interval=COMPACTION_INTERVAL, overlap_size=2,
                                                        token_threshold=COMPACTION_TOKEN_THRESHOLD, event_retention_size=5),
        graph_dir=str(DATA_DIR / 'graphs'), embeddings_dir=str(DATA_DIR / 'embeddings'),
        wheels_dir=DATA_DIR / 'wheels', verbose=False)

def record(r, task, t_start: float, t_end: float) -> dict:
    patch = r.agent_patch or ''
    dur = float(r.duration_seconds or 0)
    wall = t_end - t_start
    return {'instance_id': task.instance_id, 'repo': task.repo, 'resolved': bool(r.resolved),
            'status': getattr(r, 'status', None), 'test_exit_code': r.test_exit_code,
            'duration_s': round(dur, 1), 'task_wall_s': round(wall, 1),
            'setup_s': round(max(0.0, wall - dur), 1),   # everything outside the agent loop (setup + grading)  VERIFY
            'tool_calls': getattr(r, 'tool_calls', None), 'llm_calls': getattr(r, 'total_llm_calls', None),
            'total_tokens': getattr(r, 'total_tokens', None), 'error': getattr(r, 'error', None) or getattr(r, 'error_message', None),
            'patch_chars': len(patch), 'patch': patch, 'trace': str(getattr(r, 'trace_path', '') or getattr(r, 'trace_json_path', '') or ''),
            'test_tail': (r.test_output or '')[-1500:], 'max_time_minutes': EVAL.get('max_time_minutes'),
            'max_tool_calls': EVAL.get('max_tool_calls')}

def append_row(path: Path, row: dict) -> None:
    with path.open('a', encoding='utf-8') as fh:
        fh.write(json.dumps(row) + '\n')

In [ ]:
# ---------------------------------------------------------------- EXP-000 gold control (Q5)
# `sound` must exist even when the gold control is skipped, or the agent cell below raises NameError.
sound = list(candidates)
if RUN_GOLD_CONTROL:
    from adk_eval_core.tracing import SessionTrace  # [V] tracing/__init__.py re-exports it; no-arg ctor

    class GoldEvaluator(Evaluator):
        """Phase 1 returns the reference patch; no model call, nothing is shown to any agent."""
        async def _run_agent_sandbox(self, task, *args, **kwargs):  # [V] evaluate.py:194 -> tuple[str, str|None, SessionTrace]
            return task.patch, None, SessionTrace()

    gold = GoldEvaluator(make_config(RUN_DIR / 'gold_control'))
    gold_rows = []
    for i, t in enumerate(candidates, 1):
        t0 = time.time()
        r = run_sync(lambda: gold.evaluate_task(task=t, task_index=i, total_tasks=len(candidates)))
        row = {'instance_id': t.instance_id, 'repo': t.repo, 'mode': 'gold', 'resolved': bool(r.resolved),
               'exit': r.test_exit_code, 'status': getattr(r, 'status', None), 'seconds': round(time.time() - t0, 1), 'error': (getattr(r, 'error', '') or '')[:300],
               'tail': (r.test_output or '')[-600:]}
        gold_rows.append(row)
        append_row(RUN_DIR / 'gold_control.jsonl', row)
        print('gold', t.instance_id, 'resolved=', row['resolved'], f"{row['seconds']}s")
    failures = [x['instance_id'] for x in gold_rows if not x['resolved']]
    (RUN_DIR / 'gold_failures.txt').write_text('# gold-control failures (exclude from cohorts)\n' + '\n'.join(failures) + '\n')
    sound = [t for t, x in zip(candidates, gold_rows) if x['resolved']]
    print(f'gold control: {len(sound)}/{len(candidates)} sound; failures: {failures}')

In [ ]:
# ---------------------------------------------------------------- baseline agent eval
summary = {'exp_id': EXP_ID, 'server_start_s': SERVER_START_S, 'decode_probe': DECODE, 'eval_config': EVAL,
           'compaction': {'interval': COMPACTION_INTERVAL, 'token_threshold': COMPACTION_TOKEN_THRESHOLD},
           'tasks': [], 'resolved': 0}
if RUN_AGENT:
    ev = Evaluator(make_config(RUN_DIR / 'official'))
    for i, t in enumerate(sound, 1):
        t0 = time.time()
        r = run_sync(lambda: ev.evaluate_task(task=t, task_index=i, total_tasks=len(sound)))
        row = record(r, t, t0, time.time())
        append_row(RUN_DIR / 'task_results.jsonl', row)
        (RUN_DIR / 'patches' / f'{t.instance_id}.patch').write_text(row['patch'])
        for cand in (Path(row['trace']) if row['trace'] else None, RUN_DIR / 'official' / 'traces' / f'trace_{t.instance_id}.json'):
            if cand and cand.exists():
                shutil.copy(cand, RUN_DIR / 'traces' / f'trace_{t.instance_id}.json'); break
        summary['tasks'].append(t.instance_id); summary['resolved'] += int(row['resolved'])
        print(f"[{i}/{len(sound)}] {t.instance_id} resolved={row['resolved']} calls={row['tool_calls']} "
              f"agent={row['duration_s']}s setup={row['setup_s']}s err={(row['error'] or '')[:100]}")
    n = len(summary['tasks'])
    print(f"RESOLVED {summary['resolved']}/{n}" if n else 'no tasks run')
(RUN_DIR / 'summary.json').write_text(json.dumps(summary, indent=1))

In [ ]:
# ---------------------------------------------------------------- bundle for download + stop server
# Official logs may contain repository snippets: keep the bundle private, never commit it (plan §1.3).
for extra in (RUN_DIR / 'official' / 'logs',):
    if extra.exists():
        shutil.copytree(extra, RUN_DIR / 'logs', dirs_exist_ok=True)
shutil.make_archive(str(Path('/kaggle/working/runs') / EXP_ID), 'zip', RUN_DIR)
print('bundle:', Path('/kaggle/working/runs') / f'{EXP_ID}.zip')
try:
    server.stop()
except Exception:  # noqa: BLE001
    pass